In [1]:
import re
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ============================================================
# 0. 공통 설정: 경로는 여기서만 관리
#    v2 변경사항:
#    - 추론 입력도 학습과 동일한 전처리 사용
#    - clean_text -> normalize_conversation -> [턴] 기반 text 생성
# ============================================================
CONFIG = {
    "test_path": Path("./data/real/test.csv"),
    "sample_submission_path": Path("./data/real/submission.csv"),
    "output_dir": Path("./submission"),
    "batch_size": 32,
    "max_length": 512,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

LABEL_NAMES = ["협박", "갈취", "직장내괴롭힘", "기타괴롭힘", "일반대화"]
NUM_LABELS = len(LABEL_NAMES)
TURN_TOKEN = "[턴]"


def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# 1. 학습 노트북과 동일한 텍스트 전처리
#    - merge_preprocessing_pipeline.ipynb 기준
# ============================================================
def clean_text(text: str) -> str:
    """한국어 대화체 텍스트 정제"""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)                   # 반복문자 축소
    text = re.sub(r'([ㄱ-ㅎㅏ-ㅣ])\1{2,}', r'\1\1', text)          # 자모 반복 축소
    text = re.sub(r'[^\w\s가-힣a-zA-Z0-9.,!?~\n]', ' ', text)     # 특수문자 제거
    text = re.sub(r'[ \t]+', ' ', text)                          # 연속 공백 축소
    text = re.sub(r'\n+', '\n', text)                            # 연속 줄바꿈 축소
    return text.strip()


def normalize_conversation(text: str) -> str:
    """대화 턴 구분을 [턴] 토큰으로 변환"""
    if not isinstance(text, str):
        return ""
    turns = [t.strip() for t in text.split("\n") if t.strip()]
    return f" {TURN_TOKEN} ".join(turns)


def build_inference_texts(conversation_series: pd.Series) -> list[str]:
    """
    test.csv의 raw conversation을
    학습과 동일한 입력 스키마(text=conversation_norm)로 변환
    """
    conversation_clean = conversation_series.apply(clean_text)
    conversation_norm = conversation_clean.apply(normalize_conversation)

    empty_mask = conversation_norm.str.len().eq(0)
    if empty_mask.any():
        empty_count = int(empty_mask.sum())
        raise ValueError(f"전처리 후 빈 문자열이 발생했습니다: {empty_count}개")

    return conversation_norm.tolist()


def detect_submission_columns(sample_df: pd.DataFrame):
    cols = sample_df.columns.tolist()

    id_candidates = ["file_name", "idx", "id"]
    target_candidates = ["class", "target", "label"]

    id_col = next((c for c in id_candidates if c in cols), None)
    target_col = next((c for c in target_candidates if c in cols), None)

    if id_col is None or target_col is None:
        raise ValueError(f"sample submission 컬럼 인식 실패: {cols}")

    return id_col, target_col


def validate_inputs(test_df: pd.DataFrame, sample_df: pd.DataFrame):
    if "conversation" not in test_df.columns:
        raise ValueError("test.csv에 conversation 컬럼이 없습니다.")

    if test_df["conversation"].isna().any():
        na_count = int(test_df["conversation"].isna().sum())
        raise ValueError(f"test.csv conversation 컬럼에 결측치가 있습니다: {na_count}개")

    if len(test_df) != len(sample_df):
        raise ValueError(
            f"행 수 불일치: test.csv={len(test_df)}, sample_submission={len(sample_df)}"
        )


def validate_id_alignment(test_df: pd.DataFrame, sample_df: pd.DataFrame, id_col: str):
    if "idx" in test_df.columns:
        left = test_df["idx"].astype(str).reset_index(drop=True)
        right = sample_df[id_col].astype(str).reset_index(drop=True)

        if not left.equals(right):
            mismatches = (left != right).sum()
            raise ValueError(
                f"test.csv idx와 sample submission의 {id_col}가 일치하지 않습니다. "
                f"불일치 개수: {int(mismatches)}"
            )


def load_model_bundle(model_path: str | Path, device: str):
    model_path = str(model_path)

    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    model_num_labels = getattr(model.config, "num_labels", None)
    if model_num_labels != NUM_LABELS:
        raise ValueError(
            f"모델의 num_labels={model_num_labels} 입니다. "
            f"이 과제는 {NUM_LABELS}개 클래스 모델이어야 합니다."
        )

    if tokenizer.pad_token is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            raise ValueError("tokenizer.pad_token이 없고 대체 가능한 eos_token도 없습니다.")

    # 학습 시 [턴] special token 사용. 체크포인트 tokenizer에도 들어 있어야 함.
    added_vocab = tokenizer.get_added_vocab()
    full_vocab = tokenizer.get_vocab()
    if TURN_TOKEN not in added_vocab and TURN_TOKEN not in full_vocab:
        raise ValueError(
            f"tokenizer에 {TURN_TOKEN} special token이 없습니다. "
            "학습에 사용한 체크포인트/tokenizer 저장본인지 확인하세요."
        )

    model.to(device)
    model.eval()
    return tokenizer, model


@torch.no_grad()
def predict_labels(texts, tokenizer, model, device, batch_size=32, max_length=512):
    preds_all = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(**enc).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        preds_all.extend(preds.tolist())

    preds_all = np.array(preds_all, dtype=int)

    if len(preds_all) != len(texts):
        raise ValueError(
            f"예측 개수 불일치: preds={len(preds_all)}, texts={len(texts)}"
        )

    if not np.isin(preds_all, np.arange(NUM_LABELS)).all():
        bad_values = sorted(set(preds_all.tolist()) - set(range(NUM_LABELS)))
        raise ValueError(f"허용 범위(0~{NUM_LABELS-1}) 밖 라벨이 있습니다: {bad_values}")

    return preds_all


def make_submission(model_path: str | Path, output_name: str, config: dict = CONFIG):
    """
    model_path : 모델 폴더 경로
    output_name: 저장할 파일명만 입력
                 예) sub_roberta_focal.csv
    """
    set_seed(config["seed"])

    if not str(output_name).lower().endswith(".csv"):
        output_name = f"{output_name}.csv"

    test_df = pd.read_csv(config["test_path"])
    sample_df = pd.read_csv(config["sample_submission_path"])

    validate_inputs(test_df, sample_df)
    id_col, target_col = detect_submission_columns(sample_df)
    validate_id_alignment(test_df, sample_df, id_col)

    tokenizer, model = load_model_bundle(model_path, config["device"])

    inference_texts = build_inference_texts(test_df["conversation"])

    preds = predict_labels(
        texts=inference_texts,
        tokenizer=tokenizer,
        model=model,
        device=config["device"],
        batch_size=config["batch_size"],
        max_length=config["max_length"],
    )

    submit_df = sample_df.copy()
    submit_df[target_col] = preds

    config["output_dir"].mkdir(parents=True, exist_ok=True)
    output_path = config["output_dir"] / output_name
    submit_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("=" * 70)
    print(f"model_path : {model_path}")
    print(f"output_path: {output_path}")
    print(f"device     : {config['device']}")
    print(f"rows       : {len(submit_df)}")
    print(f"columns    : {submit_df.columns.tolist()}")
    print(f"label_dist : {submit_df[target_col].value_counts().sort_index().to_dict()}")
    print(f"sample_text: {inference_texts[0][:120] if inference_texts else 'N/A'}")
    print("=" * 70)

    return submit_df


def make_multiple_submissions(job_list, config: dict = CONFIG):
    """
    job_list 예시
    [
        {"model_path": "./checkpoint_a", "output_name": "sub_a.csv"},
        {"model_path": "./checkpoint_b", "output_name": "sub_b.csv"},
    ]
    """
    results = {}

    for job in job_list:
        if "model_path" not in job or "output_name" not in job:
            raise ValueError(f"job 항목에 model_path 또는 output_name이 없습니다: {job}")

        df = make_submission(
            model_path=job["model_path"],
            output_name=job["output_name"],
            config=config,
        )
        results[job["output_name"]] = df

    return results


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 제출 파일 만들기 (v2: 학습과 동일한 전처리 적용)
# jobs = [
#     {"model_path": "./checkpoint_baseline", "output_name": "sub_baseline_v2.csv"},
#     {"model_path": "./checkpoint_focal", "output_name": "sub_focal_v2.csv"},
# ]
#
# results = make_multiple_submissions(jobs)

jobs = [
    {"model_path": "./results_dktc_wd_0.1/checkpoint-741", "output_name": "sub_wd_0.1_v2.csv"},
]

results = make_multiple_submissions(jobs)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2025.57it/s]


model_path : ./results_dktc_wd_0.1/checkpoint-741
output_path: submission/sub_wd_0.1_v2.csv
device     : cuda
rows       : 500
columns    : ['file_name', 'class']
label_dist : {0: 114, 1: 110, 2: 114, 3: 129, 4: 33}
sample_text: 아가씨 담배한갑주소 네 4500원입니다 어 네 지갑어디갔지 에이 버스에서 잃어버렸나보네 그럼 취소할까요 아가씨 내 여기단골이니 담에 갖다줄께 저도 알바생이라 외상안됩니다 아따 누가 떼먹는다고 그러나 갖다준다고 안됩니
